# Name : Muhammad Danish Zaheer Awan
# REG id : 25280092
# Task 1 DPO

## Shared setup 

In [1]:
# Import filesystem, system, table, and notebook display utilities.
# These shared imports are used throughout the Task 1 workbench.
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

In [2]:
# Locate the repository root and make project modules importable.
# Import the Task 1 training, evaluation, analysis, and plotting helpers.
root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'configs' / 'dpo.yaml').exists())
os.chdir(root)
sys.path.insert(0, str(root))
from common.data import load_yaml, read_jsonl
from common.logging_utils import load_json
from task1_dpo.core.trainer import train_dpo
from task1_dpo.evaluate import run_evaluation
from task1_dpo.ablate_beta import run_beta_study
from task1_dpo.analyze_length import run_length_study
from task1_dpo.analysis.aggregate import dpo_summary_table, length_tables
from task1_dpo.analysis.plots import plot_training_history, plot_beta_comparison, plot_length_strata, plot_word_limit_comparison
from task1_dpo.evaluation.qualitative import review_candidates
from task1_dpo.rescore_rewards import rescore_saved_rewards
figure_dir = root / 'task1_dpo/results/figures'
figure_dir.mkdir(parents=True, exist_ok=True)
print('Repository:', root)

c:\Users\DanishZaheer\anaconda3\envs\ATML\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repository: c:\Users\DanishZaheer\Desktop\LUMS\Semester_3\ATML\PA_2


### Model cache check

In [3]:
# Load the DPO configuration and inspect the local Hugging Face cache.
# Download the policy or reward model only when it is not already cached.
from huggingface_hub import snapshot_download
from huggingface_hub.constants import HF_HUB_CACHE
from huggingface_hub.errors import LocalEntryNotFoundError

config = load_yaml('configs/dpo.yaml')
print('Hugging Face cache:', HF_HUB_CACHE)
for role, model_id in [('policy', config['base_model']), ('reward', config['reward_model'])]:
    try:
        model_path = snapshot_download(repo_id=model_id, local_files_only=True)
        print(f'{role}: already cached at {model_path}')
    except LocalEntryNotFoundError:
        print(f'{role}: downloading {model_id} ...')
        model_path = snapshot_download(repo_id=model_id)
        print(f'{role}: downloaded to {model_path}')

Hugging Face cache: C:\Users\DanishZaheer\.cache\huggingface\hub
policy: already cached at C:\Users\DanishZaheer\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct\snapshots\989aa7980e4cf806f80c7fef2b1adb7bc71aa306
reward: already cached at C:\Users\DanishZaheer\.cache\huggingface\hub\models--yavuz-ai--qwen2.5-1.5b-rm-ultrafeedback\snapshots\f98759a4a1ccdc47a3136748b6bfea9a32ba8fcc


### Fixed data check

In [4]:
# Read each fixed Task 1 dataset and print its number of examples.
# Also summarize the available length strata for the length-bias study.
from collections import Counter
for name in ('dpo_standard_train', 'dpo_standard_eval', 'dpo_length_balanced_train', 'dpo_length_stratified_eval', 'word_limit_prompts'):
    rows = read_jsonl(f'data/{name}.jsonl')
    counts = Counter(row.get('length_stratum') for row in rows if 'length_stratum' in row)
    print(name, len(rows), dict(counts))

dpo_standard_train 1500 {}
dpo_standard_eval 300 {}
dpo_length_balanced_train 1500 {'rejected_longer': 500, 'preferred_longer': 500, 'length_matched': 500}
dpo_length_stratified_eval 246 {'rejected_longer': 82, 'preferred_longer': 82, 'length_matched': 82}
word_limit_prompts 10 {}


## Task 1 - Part 1: Standard DPO

In [5]:
# Train standard DPO only when its saved adapter is unavailable.
# Run or reuse the corresponding held-out evaluation metrics afterward.
standard_adapter = root / 'task1_dpo/outputs/standard/adapter_config.json'
if not standard_adapter.exists():
    display(train_dpo())
else:
    print('Loaded existing standard adapter:', standard_adapter)
standard_metrics = root / 'task1_dpo/results/standard/eval_metrics.json'
if not standard_metrics.exists():
    display(run_evaluation())
else:
    display(load_json(standard_metrics))

Loaded existing standard adapter: c:\Users\DanishZaheer\Desktop\LUMS\Semester_3\ATML\PA_2\task1_dpo\outputs\standard\adapter_config.json


{'run_name': 'standard',
 'adapter': 'C:\\Users\\DanishZaheer\\Desktop\\LUMS\\Semester_3\\ATML\\PA_2\\task1_dpo\\outputs\\standard',
 'adapter_sha256': 'c920d63d334eb5702e3f3e24e804267d1592c50318bb893f5bc182736386ca56',
 'eval_data': 'C:\\Users\\DanishZaheer\\Desktop\\LUMS\\Semester_3\\ATML\\PA_2\\data\\dpo_standard_eval.jsonl',
 'beta': 0.1,
 'seed': 6304,
 'generation': {'temperature': 0.7, 'top_p': 0.9, 'do_sample': True},
 'max_generation_tokens': 256,
 'heldout_dpo_loss': 0.6766497317949931,
 'preference_accuracy': 0.6466666666666666,
 'preference_pairs': 300,
 'generated_prompts': 300,
 'sampled_kl_token_mean': 9.14773203182714e-05,
 'response_length_mean': 154.67,
 'response_length_std': 105.24403910277611,
 'response_length_iqr': 213.5,
 'truncated_fraction': 0.4533333333333333,
 'reward_model_score_mean': 1.2022842534383138,
 'reference_reward_score_mean': 1.2074203236897787,
 'reward_scoring': {'reward_model': 'yavuz-ai/qwen2.5-1.5b-rm-ultrafeedback',
  'reward_tokenizer': 'Q

In [6]:
# Plot the saved standard-DPO training history for visual inspection.
# Save the figure in the committed results directory and display it here.
figure_dir = root / 'task1_dpo/results/figures'
figure_dir.mkdir(parents=True, exist_ok=True)
figure = plot_training_history()
figure.savefig(figure_dir / 'standard_training.png', dpi=180, bbox_inches='tight')
display(figure)

<Figure size 1000x350 with 2 Axes>

In [7]:
# Load standard-run generations and select useful qualitative candidates.
# Print each prompt with its DPO and reference responses for comparison.
generations = read_jsonl('task1_dpo/results/standard/generations.jsonl')
candidates = review_candidates(generations, limit=10)
for item in candidates:
    print('\nPROMPT ID:', item['prompt_id'], 'REWARD CHANGE:', round(item['reward_change'], 3))
    print('PROMPT:', item['prompt'][:500])
    print('DPO:', item['policy_response'][:900])
    print('REFERENCE:', item['reference_response'][:900])


PROMPT ID: 3ae6b979253533ae24823844cd8af57ef3fb1d4f23c4faf0546c3335a7be0e5f REWARD CHANGE: 2.723
PROMPT: Given the sentence "Five young lady getting ready to run the 100m dash." is it true that "A group of women prepare to race."?
Let's solve step-by-step:
DPO: Step 1: Analyze the given sentence.
"Five young ladies getting ready to run the 100m dash."
- There are five people (ladies)
- They are young
- They are preparing for a race (100m dash)
- The gender of the individuals is specified as "young ladies"

Step 2: Compare with the statement we need to verify.
"A group of women prepare to race."

- A group refers to multiple people
- Women is the term used instead of "young ladies"
- Prepare to race means they are getting ready for a race

Step 3: Evaluate if all elements in the given sentence match or can be reasonably inferred from the statement.

The key points:
- There are multiple females (five young ladies = a group of women)
- They are getting ready to participate in a race (the

## Task 1 - Part 2: Beta study

In [9]:
# Run or reuse the three short beta-ablation conditions and show their metrics.
# Build the comparison table and save the beta-effect visualization.
display(pd.DataFrame(run_beta_study()))
display(dpo_summary_table())
figure_dir = root / 'task1_dpo/results/figures'
figure_dir.mkdir(parents=True, exist_ok=True)
figure = plot_beta_comparison()
figure.savefig(figure_dir / 'beta_comparison.png', dpi=180, bbox_inches='tight')
display(figure)

,run_name,adapter,adapter_sha256,eval_data,beta,seed,generation,max_generation_tokens,heldout_dpo_loss,preference_accuracy,...,sampled_kl_token_mean,response_length_mean,response_length_std,response_length_iqr,truncated_fraction,reward_model_score_mean,reference_reward_score_mean,reward_scoring,reward_model_score_std,reference_reward_score_std
0,beta_0.03,C:\Users\DanishZaheer\Desktop\LUMS\Semester_3\...,a058a818f9327a8f264f423d16abfaee38de64eb03091a...,C:\Users\DanishZaheer\Desktop\LUMS\Semester_3\...,0.03,6304,"{'temperature': 0.7, 'top_p': 0.9, 'do_sample'...",256,0.691292,0.610000,...,0.000087,155.156667,105.631145,217.0,0.453333,1.176968,1.20742,{'reward_model': 'yavuz-ai/qwen2.5-1.5b-rm-ult...,1.496006,1.476339
1,beta_0.10,C:\Users\DanishZaheer\Desktop\LUMS\Semester_3\...,c734883c5a4f4e3eb324c89a386a92ed66c24488d5dc13...,C:\Users\DanishZaheer\Desktop\LUMS\Semester_3\...,0.10,6304,"{'temperature': 0.7, 'top_p': 0.9, 'do_sample'...",256,0.687365,0.596667,...,0.000086,155.410000,105.881956,217.0,0.460000,1.153343,1.20742,{'reward_model': 'yavuz-ai/qwen2.5-1.5b-rm-ult...,1.466851,1.476339
2,beta_0.30,C:\Users\DanishZaheer\Desktop\LUMS\Semester_3\...,e5ca7c5935cab5e29831383a3888b7d44530ea743b9e6f...,C:\Users\DanishZaheer\Desktop\LUMS\Semester_3\...,0.30,6304,"{'temperature': 0.7, 'top_p': 0.9, 'do_sample'...",256,0.680745,0.620000,...,0.000060,155.720000,105.768591,217.0,0.453333,1.174160,1.20742,{'reward_model': 'yavuz-ai/qwen2.5-1.5b-rm-ult...,1.489364,1.476339


,run,training_budget,training_examples,optimizer_updates,beta,heldout_pairs,heldout_dpo_loss,preference_accuracy,sampled_kl_token_mean,reward_model_score_mean,response_length_mean,response_length_std,response_length_iqr,truncated_fraction
0,standard,one epoch,1500,94,0.10,300,0.676650,0.646667,0.000091,1.202284,154.670000,105.244039,213.5,0.453333
1,beta_0.03,short 600-pair fork,600,38,0.03,300,0.691292,0.610000,0.000087,1.176968,155.156667,105.631145,217.0,0.453333
2,beta_0.10,short 600-pair fork,600,38,0.10,300,0.687365,0.596667,0.000086,1.153343,155.410000,105.881956,217.0,0.460000
3,beta_0.30,short 600-pair fork,600,38,0.30,300,0.680745,0.620000,0.000060,1.174160,155.720000,105.768591,217.0,0.453333


<Figure size 1000x700 with 4 Axes>

In [ ]:
# Rescore all saved responses independently with the fixed reward model.
# This refreshes reward metrics without repeating training or generation.
corrected_metrics = rescore_saved_rewards()
print('Independent reward scoring is complete for:', ', '.join(corrected_metrics))

## Task 1 - Part 3: Length-bias study

In [10]:
# Run or reuse the length-balanced study and display its diagnostic tables.
# Save plots comparing length strata and word-limit compliance behavior.
display(run_length_study())
strata, word_limits = length_tables()
display(pd.read_csv('task1_dpo/results/length/dataset_length_metrics.csv'), strata, word_limits)
figure_dir = root / 'task1_dpo/results/figures'
figure_dir.mkdir(parents=True, exist_ok=True)
figure = plot_length_strata()
figure.savefig(figure_dir / 'length_strata.png', dpi=180, bbox_inches='tight')
display(figure)
word_figure = plot_word_limit_comparison()
word_figure.savefig(figure_dir / 'word_limit_comparison.png', dpi=180, bbox_inches='tight')
display(word_figure)

Using completed length-study results; skipping evaluation.


{'dataset_lengths': [{'dataset': 'standard',
   'pairs': 1500,
   'chosen_tokens_mean': 282.11466666666666,
   'rejected_tokens_mean': 239.66466666666668,
   'token_difference_mean': 42.45,
   'chosen_longer_fraction': 0.5626666666666666,
   'rejected_longer_fraction': 0.422,
   'exact_length_tie_fraction': 0.015333333333333332},
  {'dataset': 'length_balanced',
   'pairs': 1500,
   'chosen_tokens_mean': 156.85533333333333,
   'rejected_tokens_mean': 151.692,
   'token_difference_mean': 5.163333333333333,
   'chosen_longer_fraction': 0.49133333333333334,
   'rejected_longer_fraction': 0.486,
   'exact_length_tie_fraction': 0.02266666666666667}],
 'strata': [{'policy': 'standard',
   'length_stratum': 'preferred_longer',
   'pairs': 82,
   'preference_accuracy': 0.4634146341463415,
   'mean_preference_margin': -0.10950339712747713},
  {'policy': 'standard',
   'length_stratum': 'length_matched',
   'pairs': 82,
   'preference_accuracy': 0.5853658536585366,
   'mean_preference_margin': 0

,dataset,pairs,chosen_tokens_mean,rejected_tokens_mean,token_difference_mean,chosen_longer_fraction,rejected_longer_fraction,exact_length_tie_fraction
0,standard,1500,282.114667,239.664667,42.450000,0.562667,0.422,0.015333
1,length_balanced,1500,156.855333,151.692000,5.163333,0.491333,0.486,0.022667


,policy,length_stratum,pairs,preference_accuracy,mean_preference_margin
0,standard,preferred_longer,82,0.463415,-0.109503
1,standard,length_matched,82,0.585366,0.205282
2,standard,rejected_longer,82,0.658537,0.716339
3,length_balanced,preferred_longer,82,0.414634,-0.506560
4,length_balanced,length_matched,82,0.597561,0.376584
5,length_balanced,rejected_longer,82,0.780488,1.464291


,policy,prompts,response_length_mean,response_length_std,word_limit_compliance
0,standard,10,50.7,23.013257,0.5
1,length_balanced,10,49.5,16.070159,0.3


<Figure size 800x350 with 1 Axes>

<Figure size 900x350 with 2 Axes>

### Part 3 response review

In [11]:
# Display per-prompt word-limit statistics from the standard policy.
# Print standard and length-balanced responses for qualitative comparison.
word_generations = read_jsonl('task1_dpo/results/length/standard_word_generations.jsonl')
display(pd.DataFrame(word_generations)[['prompt_id', 'response_length', 'policy_word_limit_compliance', 'reference_word_limit_compliance']])
for item in read_jsonl('task1_dpo/results/length/qualitative_examples.jsonl'):
    print('\nWORD-LIMIT PROMPT:', item['prompt_id'], item['prompt'])
    print('STANDARD:', item['standard_response'][:700])
    print('BALANCED:', item['balanced_response'][:700])

,prompt_id,response_length,policy_word_limit_compliance,reference_word_limit_compliance
0,wl01,41,1.0,1.0
1,wl02,34,0.0,0.0
2,wl03,26,1.0,1.0
3,wl04,58,0.0,0.0
4,wl05,49,1.0,0.0
5,wl06,31,1.0,0.0
6,wl07,68,0.0,0.0
7,wl08,60,0.0,0.0
8,wl09,33,1.0,1.0
9,wl10,107,0.0,0.0



WORD-LIMIT PROMPT: wl01 Explain why regularization can improve generalization in at most 40 words.
STANDARD: Regularization helps prevent overfitting by adding a penalty term to the loss function that discourages complex models from fitting noise in the training data, thus improving model's ability to generalize to unseen data.
BALANCED: Regularization helps prevent overfitting by adding a penalty term to the loss function that discourages complex models from fitting noise in the training data, thus improving model's ability to generalize to unseen data.

WORD-LIMIT PROMPT: wl02 Define overfitting in no more than 25 words.
STANDARD: Overfitting is when a model learns the training data too well, capturing noise and outliers instead of general patterns, leading to poor performance on new, unseen data.
BALANCED: Overfitting is when a model learns the training data too well, capturing noise and outliers instead of general patterns, leading to poor performance on new, unseen data.

WORD-LI

## DPO summary and beta research 

In [12]:
# Assemble the standard and beta-run metrics needed for the report.
# Display the summary tables and recreate the main training and beta figures.
summary = dpo_summary_table()
display(summary.round(6))
beta_only = summary.loc[summary['run'] != 'standard', [
    'beta', 'heldout_dpo_loss', 'preference_accuracy',
    'sampled_kl_token_mean', 'reward_model_score_mean',
    'response_length_mean', 'response_length_std', 'truncated_fraction'
]].sort_values('beta')
display(beta_only.round(6))
for filename, make_figure in (
    ('standard_training.png', plot_training_history),
    ('beta_comparison.png', plot_beta_comparison),
):
    figure = make_figure()
    figure.savefig(figure_dir / filename, dpi=180, bbox_inches='tight')
    display(figure)

,run,training_budget,training_examples,optimizer_updates,beta,heldout_pairs,heldout_dpo_loss,preference_accuracy,sampled_kl_token_mean,reward_model_score_mean,response_length_mean,response_length_std,response_length_iqr,truncated_fraction
0,standard,one epoch,1500,94,0.10,300,0.676650,0.646667,0.000091,1.202284,154.670000,105.244039,213.5,0.453333
1,beta_0.03,short 600-pair fork,600,38,0.03,300,0.691292,0.610000,0.000087,1.176968,155.156667,105.631145,217.0,0.453333
2,beta_0.10,short 600-pair fork,600,38,0.10,300,0.687365,0.596667,0.000086,1.153343,155.410000,105.881956,217.0,0.460000
3,beta_0.30,short 600-pair fork,600,38,0.30,300,0.680745,0.620000,0.000060,1.174160,155.720000,105.768591,217.0,0.453333


,beta,heldout_dpo_loss,preference_accuracy,sampled_kl_token_mean,reward_model_score_mean,response_length_mean,response_length_std,truncated_fraction
1,0.03,0.691292,0.610000,0.000087,1.176968,155.156667,105.631145,0.453333
2,0.10,0.687365,0.596667,0.000086,1.153343,155.410000,105.881956,0.460000
3,0.30,0.680745,0.620000,0.000060,1.174160,155.720000,105.768591,0.453333


<Figure size 1000x350 with 2 Axes>

<Figure size 1000x700 with 4 Axes>

### length-confounding research 

In [13]:
# Display dataset-length, stratum, and word-limit evidence tables.
# Recreate the two figures supporting the length-confounding analysis.
dataset_lengths = pd.read_csv('task1_dpo/results/length/dataset_length_metrics.csv')
strata, word_limits = length_tables()
display(dataset_lengths.round(4), strata.round(4), word_limits.round(4))
for filename, make_figure in (
    ('length_strata.png', plot_length_strata),
    ('word_limit_comparison.png', plot_word_limit_comparison),
):
    figure = make_figure()
    figure.savefig(figure_dir / filename, dpi=180, bbox_inches='tight')
    display(figure)

,dataset,pairs,chosen_tokens_mean,rejected_tokens_mean,token_difference_mean,chosen_longer_fraction,rejected_longer_fraction,exact_length_tie_fraction
0,standard,1500,282.1147,239.6647,42.4500,0.5627,0.422,0.0153
1,length_balanced,1500,156.8553,151.6920,5.1633,0.4913,0.486,0.0227


,policy,length_stratum,pairs,preference_accuracy,mean_preference_margin
0,standard,preferred_longer,82,0.4634,-0.1095
1,standard,length_matched,82,0.5854,0.2053
2,standard,rejected_longer,82,0.6585,0.7163
3,length_balanced,preferred_longer,82,0.4146,-0.5066
4,length_balanced,length_matched,82,0.5976,0.3766
5,length_balanced,rejected_longer,82,0.7805,1.4643


,policy,prompts,response_length_mean,response_length_std,word_limit_compliance
0,standard,10,50.7,23.0133,0.5
1,length_balanced,10,49.5,16.0702,0.3


<Figure size 800x350 with 1 Axes>

<Figure size 900x350 with 2 Axes>

### qualitative research 

In [14]:
# Combine qualitative candidates from the standard and all beta runs.
# Rank reward changes and print the strongest policy-reference comparisons.
candidate_files = {
    'standard': 'task1_dpo/results/standard/qualitative_candidates.jsonl',
    'beta_0.03': 'task1_dpo/results/beta/beta_0.03/qualitative_candidates.jsonl',
    'beta_0.10': 'task1_dpo/results/beta/beta_0.10/qualitative_candidates.jsonl',
    'beta_0.30': 'task1_dpo/results/beta/beta_0.30/qualitative_candidates.jsonl',
}
candidate_rows = []
for run_name, path in candidate_files.items():
    for row in read_jsonl(path):
        candidate_rows.append({'run': run_name, **row})
candidate_frame = pd.DataFrame(candidate_rows).sort_values('reward_change', ascending=False)
display(candidate_frame[[
    'run', 'prompt_id', 'policy_reward_score', 'reference_reward_score',
    'reward_change', 'length_change', 'manual_quality_assessment'
]].head(20).round(4))
for _, item in candidate_frame.head(8).iterrows():
    print('\nRUN:', item['run'], 'PROMPT ID:', item['prompt_id'])
    print('REWARD CHANGE:', round(item['reward_change'], 4))
    print('PROMPT:', item['prompt'][:700])
    print('DPO:', item['policy_response'][:1000])
    print('REFERENCE:', item['reference_response'][:1000])

,run,prompt_id,policy_reward_score,reference_reward_score,reward_change,length_change,manual_quality_assessment
0,standard,3ae6b979253533ae24823844cd8af57ef3fb1d4f23c4fa...,3.2402,0.5171,2.7231,-23,None
60,beta_0.30,3ae6b979253533ae24823844cd8af57ef3fb1d4f23c4fa...,3.2402,0.5171,2.7231,-23,None
40,beta_0.10,3ae6b979253533ae24823844cd8af57ef3fb1d4f23c4fa...,3.2402,0.5171,2.7231,-23,None
20,beta_0.03,3ae6b979253533ae24823844cd8af57ef3fb1d4f23c4fa...,3.2402,0.5171,2.7231,-23,None
1,standard,be262df45f04ab93b838664a7235f420fc8406c73e0af9...,1.5137,-1.0234,2.5371,-13,None
21,beta_0.03,ba7ddaf72f0e034da39b43276b9f418894a12a6fffdbcf...,2.9141,0.4121,2.5020,0,None
2,standard,21a37cd9a72739d56d90061c42121b8f2688c5a4a623d3...,2.2715,-0.1455,2.4170,3,None
3,standard,d70053f4375dd47082610320bf3c8ab0ecfdda3ff25309...,1.0059,-1.4023,2.4082,224,None
22,beta_0.03,ae12de1584037125ac418a027b79f5a33612a8d208da40...,3.5117,1.3428,2.1689,0,None
23,beta_0.03,bdfcf969f2a84458b2498e0acc198d3094334b81d595fe...,2.0352,-0.0142,2.0493,0,None



RUN: standard PROMPT ID: 3ae6b979253533ae24823844cd8af57ef3fb1d4f23c4faf0546c3335a7be0e5f
REWARD CHANGE: 2.7231
PROMPT: Given the sentence "Five young lady getting ready to run the 100m dash." is it true that "A group of women prepare to race."?
Let's solve step-by-step:
DPO: Step 1: Analyze the given sentence.
"Five young ladies getting ready to run the 100m dash."
- There are five people (ladies)
- They are young
- They are preparing for a race (100m dash)
- The gender of the individuals is specified as "young ladies"

Step 2: Compare with the statement we need to verify.
"A group of women prepare to race."

- A group refers to multiple people
- Women is the term used instead of "young ladies"
- Prepare to race means they are getting ready for a race

Step 3: Evaluate if all elements in the given sentence match or can be reasonably inferred from the statement.

The key points:
- There are multiple females (five young ladies = a group of women)
- They are getting ready to participate

In [15]:
# Find qualitative examples where length balancing changes compliance.
# Display their metadata and print both responses for manual assessment.
length_examples = pd.DataFrame(read_jsonl('task1_dpo/results/length/qualitative_examples.jsonl'))
different_compliance = length_examples.loc[
    length_examples['standard_compliance'] != length_examples['balanced_compliance']
]
display(different_compliance[[
    'prompt_id', 'prompt', 'standard_length', 'balanced_length',
    'standard_compliance', 'balanced_compliance', 'manual_quality_assessment'
]])
for _, item in different_compliance.iterrows():
    print('\nPROMPT:', item['prompt_id'], item['prompt'])
    print('STANDARD:', item['standard_response'])
    print('BALANCED:', item['balanced_response'])

,prompt_id,prompt,standard_length,balanced_length,standard_compliance,balanced_compliance,manual_quality_assessment
4,wl05,Describe one advantage and one limitation of a...,49,67,1.0,0.0,None
5,wl06,What is distribution shift? Answer in no more ...,31,37,1.0,0.0,None



PROMPT: wl05 Describe one advantage and one limitation of attention in at most 45 words.
STANDARD: Advantage: Attention allows models to focus on the most relevant information, improving efficiency and accuracy.
Limitation: It can lead to overfitting if not managed carefully, as models may start giving more weight to irrelevant or noisy data points.
BALANCED: **Advantage:** Attention mechanisms allow models to focus on relevant parts of the input data, enhancing their ability to handle complex tasks like language translation or image classification.

**Limitation:** Attention can be computationally expensive as it requires additional parameters for each element in the model's architecture, potentially slowing down training times and increasing memory usage.

PROMPT: wl06 What is distribution shift? Answer in no more than 30 words.
STANDARD: Distribution shift refers to changes in data distributions between training and testing phases due to factors such as new users or environments, l